## 7. 核心技能：怎么读消息流

> 来源：[Agent SDK reference - Python](https://code.claude.com/docs/en/agent-sdk/python)

新手最容易卡的往往不是怎么发任务，而是**看不懂 `async for` 里一个接一个飞出来的对象**。先建立总的画面：消息流分两层，外层是 **Message**（一整条消息），一条 Message 里再装若干 **ContentBlock**（消息的内容块）。判断某个对象是哪一种，Python 里一律用 `isinstance()`。消息类都从 `claude_agent_sdk` 顶层导入，只有 `StreamEvent` 例外——官方指定从 `claude_agent_sdk.types` 导入（顶层也 re-export 了，照样能用，见 §17「流式输出」）。

这些对象看着杂，其实只分两层，记住这两层就不会乱。

**第一层看 `type` 字段**，它决定一条消息属于六种里的哪一种：系统事件、Claude 的输出、用户输入、逐 token 的增量、限流通知、任务收尾。

**第二层只针对系统事件**：`type` 为 `system` 的消息，再用一个 `subtype` 说清楚到底发生了什么事。其中最常打交道的几种（比如 hook 被触发、后台任务启动或结束），SDK 已经分别做成了专门的类——它们都是 `SystemMessage` 的**子类**，想要的字段直接是属性、伸手就拿，`isinstance(msg, SystemMessage)` 也能把整个家族一网接住；剩下不常见的，统一归到通用的 `SystemMessage`。

还有一条得提前知道：遇到不认识的消息类型，SDK 直接跳过、不报错——这样新版 CLI 配旧版 SDK 也不会崩。反过来，自己写的处理代码也要留这个余地：别假设流里只会出现见过的类型，冒出个陌生的也得兜得住。

### 7.1 Message 六种主类型

先看四种高频类型在一次真实运行里的样子。§5「最小可运行示例」 那次 `query(prompt="Read the current project and summarize the main modules.", ...)` 的消息流按序节选（真实输出，长字段截断）：

In [ ]:
HookEventMessage(
    subtype="hook_started",
    data={
        "type": "system",
        "subtype": "hook_started",
        "hook_id": "f8b634f7-9691-47fe-adf0-a4aed8505701",
        "hook_name": "SessionStart:startup",
        "hook_event": "SessionStart",
        "uuid": "149b34ea-c8e4-428b-ad6c-4b107bdc2064",
        "session_id": "d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    },
    hook_event_name="SessionStart",
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="149b34ea-c8e4-428b-ad6c-4b107bdc2064",
)
HookEventMessage(
    subtype="hook_response",
    data={
        "type": "system",
        "subtype": "hook_response",
        "hook_id": "f8b634f7-9691-47fe-adf0-a4aed8505701",
        "hook_name": "SessionStart:startup",
        "hook_event": "SessionStart",
        "output": "",
        "stdout": "",
        "stderr": "",
        "exit_code": 0,
        "outcome": "success",
        "uuid": "ed9bf792-9309-4ef4-a07d-299418d04e99",
        "session_id": "d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    },
    hook_event_name="SessionStart",
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="ed9bf792-9309-4ef4-a07d-299418d04e99",
)
# 首条永远是 SystemMessage，subtype="init"：session 元数据全在 data dict 里
SystemMessage(
    subtype="init",
    data={
        "type": "system",
        "subtype": "init",
        "cwd": "/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk",
        "session_id": "d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
        "tools": [
            "Task",
            "Bash",
            "CronCreate",
            "CronDelete",
            "CronList",
            "Edit",
            "EnterWorktree",
            "ExitWorktree",
            "Glob",
            "Grep",
            "ListMcpResourcesTool",
            "NotebookEdit",
            "Read",
            "ReadMcpResourceDirTool",
            "ReadMcpResourceTool",
            "ScheduleWakeup",
            "SendMessage",
            "Skill",
            "TaskCreate",
            "TaskGet",
            "TaskList",
            "TaskOutput",
            "TaskStop",
            "TaskUpdate",
            "WaitForMcpServers",
            "WebFetch",
            "WebSearch",
            "Workflow",
            "Write",
            "mcp__langfuse-docs__getLangfuseDocsPage",
            "mcp__langfuse-docs__getLangfuseOverview",
            "mcp__langfuse-docs__searchLangfuseDocs",
            "mcp__notion__notion-create-attachment",
            "mcp__notion__notion-create-comment",
            "mcp__notion__notion-create-database",
            "mcp__notion__notion-create-pages",
            "mcp__notion__notion-create-view",
            "mcp__notion__notion-download-attachment",
            "mcp__notion__notion-duplicate-page",
            "mcp__notion__notion-fetch",
            "mcp__notion__notion-get-async-task",
            "mcp__notion__notion-get-comments",
            "mcp__notion__notion-get-teams",
            "mcp__notion__notion-get-users",
            "mcp__notion__notion-move-pages",
            "mcp__notion__notion-query-data-sources",
            "mcp__notion__notion-query-database-view",
            "mcp__notion__notion-query-meeting-notes",
            "mcp__notion__notion-search",
            "mcp__notion__notion-update-data-source",
            "mcp__notion__notion-update-page",
            "mcp__notion__notion-update-view",
            "mcp__os-prod-apse1__cluster_health",
            "mcp__os-prod-apse1__count",
            "mcp__os-prod-apse1__get_index_mapping",
            "mcp__os-prod-apse1__list_indices",
            "mcp__os-prod-apse1__raw_api",
            "mcp__os-prod-apse1__search",
            "mcp__os-prod-cn__cluster_health",
            "mcp__os-prod-cn__count",
            "mcp__os-prod-cn__get_index_mapping",
            "mcp__os-prod-cn__list_indices",
            "mcp__os-prod-cn__raw_api",
            "mcp__os-prod-cn__search",
            "mcp__os-prod-euc1__cluster_health",
            "mcp__os-prod-euc1__count",
            "mcp__os-prod-euc1__get_index_mapping",
            "mcp__os-prod-euc1__list_indices",
            "mcp__os-prod-euc1__raw_api",
            "mcp__os-prod-euc1__search",
            "mcp__os-prod-usw2__cluster_health",
            "mcp__os-prod-usw2__count",
            "mcp__os-prod-usw2__get_index_mapping",
            "mcp__os-prod-usw2__list_indices",
            "mcp__os-prod-usw2__raw_api",
            "mcp__os-prod-usw2__search",
            "mcp__os-staging-apne1__cluster_health",
            "mcp__os-staging-apne1__count",
            "mcp__os-staging-apne1__get_index_mapping",
            "mcp__os-staging-apne1__list_indices",
            "mcp__os-staging-apne1__raw_api",
            "mcp__os-staging-apne1__search",
            "mcp__os-staging-cnnw1__cluster_health",
            "mcp__os-staging-cnnw1__count",
            "mcp__os-staging-cnnw1__get_index_mapping",
            "mcp__os-staging-cnnw1__list_indices",
            "mcp__os-staging-cnnw1__raw_api",
            "mcp__os-staging-cnnw1__search",
            "mcp__os-staging-usw2__cluster_health",
            "mcp__os-staging-usw2__count",
            "mcp__os-staging-usw2__get_index_mapping",
            "mcp__os-staging-usw2__list_indices",
            "mcp__os-staging-usw2__raw_api",
            "mcp__os-staging-usw2__search",
        ],
        "mcp_servers": [
            {"name": "os-prod-ec2", "status": "pending"},
            {"name": "os-prod-apse1", "status": "connected"},
            {"name": "postman", "status": "needs-auth"},
            {"name": "feishu-mcp", "status": "failed"},
            {"name": "langfuse-docs", "status": "connected"},
            {"name": "notion", "status": "connected"},
            {"name": "plaud", "status": "failed"},
            {"name": "os-prod-euc1", "status": "connected"},
            {"name": "os-prod-cn", "status": "connected"},
            {"name": "os-staging-usw2", "status": "connected"},
            {"name": "os-staging-apne1", "status": "connected"},
            {"name": "docs-langchain", "status": "pending"},
            {"name": "os-prod-usw2", "status": "connected"},
            {"name": "os-staging-cnnw1", "status": "connected"},
            {"name": "grafana", "status": "pending"},
            {"name": "drawio", "status": "pending"},
            {"name": "os-prod-apne1", "status": "pending"},
            {"name": "grafana-cn", "status": "pending"},
        ],
        "model": "claude-fable-5",
        "permissionMode": "default",
        "slash_commands": [
            "ai-learning",
            "algorithm-playbook",
            "archimate",
            "architecture",
            "architecture-diagram",
            "bpmn",
            "canvas",
            "clean-transcript",
            "cloud",
            "data-analytics",
            "fireworks-tech-graph",
            "graphviz",
            "icon-retrieval",
            "infocard",
            "infographic",
            "infographic-creator",
            "iot",
            "kami",
            "lark-approval",
            "lark-apps",
            "lark-attendance",
            "lark-base",
            "lark-calendar",
            "lark-contact",
            "lark-doc",
            "lark-drive",
            "lark-event",
            "lark-im",
            "lark-mail",
            "lark-markdown",
            "lark-minutes",
            "lark-note",
            "lark-okr",
            "lark-openapi-explorer",
            "lark-shared",
            "lark-sheets",
            "lark-skill-maker",
            "lark-slides",
            "lark-task",
            "lark-vc",
            "lark-vc-agent",
            "lark-whiteboard",
            "lark-wiki",
            "lark-workflow-meeting-summary",
            "lark-workflow-standup-report",
            "learning-generation-notes",
            "markitdown",
            "mindmap",
            "mubu-mindmap",
            "network",
            "security",
            "study-note-spec",
            "text-to-diagrams",
            "uml",
            "vega",
            "weekly-report",
            "yunxiao",
            "gen-commit",
            "deep-research",
            "update-config",
            "verify",
            "debug",
            "code-review",
            "simplify",
            "batch",
            "fewer-permission-prompts",
            "loop",
            "claude-api",
            "run",
            "run-skill-generator",
            "clear",
            "compact",
            "config",
            "context",
            "heapdump",
            "init",
            "reload-skills",
            "review",
            "security-review",
            "usage",
            "insights",
            "goal",
            "team-onboarding",
        ],
        "apiKeySource": "ANTHROPIC_API_KEY",
        "claude_code_version": "2.1.191",
        "output_style": "default",
        "agents": ["claude", "Explore", "general-purpose", "Plan", "statusline-setup"],
        "skills": [
            "ai-learning",
            "algorithm-playbook",
            "archimate",
            "architecture",
            "architecture-diagram",
            "bpmn",
            "canvas",
            "clean-transcript",
            "cloud",
            "data-analytics",
            "fireworks-tech-graph",
            "graphviz",
            "icon-retrieval",
            "infocard",
            "infographic",
            "infographic-creator",
            "iot",
            "kami",
            "lark-approval",
            "lark-apps",
            "lark-attendance",
            "lark-base",
            "lark-calendar",
            "lark-contact",
            "lark-doc",
            "lark-drive",
            "lark-event",
            "lark-im",
            "lark-mail",
            "lark-markdown",
            "lark-minutes",
            "lark-note",
            "lark-okr",
            "lark-openapi-explorer",
            "lark-shared",
            "lark-sheets",
            "lark-skill-maker",
            "lark-slides",
            "lark-task",
            "lark-vc",
            "lark-vc-agent",
            "lark-whiteboard",
            "lark-wiki",
            "lark-workflow-meeting-summary",
            "lark-workflow-standup-report",
            "learning-generation-notes",
            "markitdown",
            "mindmap",
            "mubu-mindmap",
            "network",
            "security",
            "study-note-spec",
            "text-to-diagrams",
            "uml",
            "vega",
            "weekly-report",
            "yunxiao",
            "deep-research",
            "update-config",
            "verify",
            "debug",
            "code-review",
            "simplify",
            "batch",
            "fewer-permission-prompts",
            "loop",
            "claude-api",
            "run",
            "run-skill-generator",
        ],
        "plugins": [],
        "analytics_disabled": True,
        "product_feedback_disabled": False,
        "uuid": "ea35c2b8-a9e6-4863-860d-40db4c6b8337",
        "memory_paths": {
            "auto": "/Users/liangzhu/.claude/projects/-Users-liangzhu-Library-Mobile-Documents-iCloud-md-obsidian-Documents-study-notes/memory/"
        },
        "fast_mode_state": "off",
    },
)
# Claude 的每轮输出是 AssistantMessage，content 里装 ContentBlock（见 §7.4「ContentBlock 四种块」）
AssistantMessage(
    content=[
        ThinkingBlock(
            thinking="",
            signature="CAISyAIKZAgPEAIYAipAfipXtio1e2z1WA8DuWPV0CsHxozc1AmVlE+/7OjzizL9dZfyCEb9d8fH8pUnByr47x4XryzpIw5v9UTH3iTHjjIOY2xhdWRlLWZhYmxlLTU4AUIIdGhpbmtpbmcSDE5uE0ryuyqRD1zcjBoMj8tyl+usPyqwO5c9IjClRk/wuzsxFk/s+GHctqXzRHPIqewhs+MvTPsJ+xZeJouUJZeuRrKWLHyW1ewqScAqkQGjcptRume+AT+vmnSjjCMZ8hvrDTM6W3zuPJ1ZBp4l9NnpfKKd2yKzaucizSLj+hVcljNzd4CUcjQkQpcNHegaUyORJAKTuaf5UG44dynQR4wLO6bSLz1nflvuahM2MR5cF62AnZE1ChOdiPKKC1cIwYE9ZEoGmi/Y+9FpKddyPlhIu3IFaA288xHk8VhML3d2GAE=",
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 2891,
        "cache_creation_input_tokens": 0,
        "cache_read_input_tokens": 68750,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 0,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 2,
    },
    message_id="msg_vrtx_012F3QXi1KiJ1Pqw9VUY29sX",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="63bda335-2dae-42bc-a565-2ec72a0df498",
)
AssistantMessage(
    content=[
        ToolUseBlock(
            id="toolu_vrtx_01R1SvMKn9DEfV2p6jZ8xFb6",
            name="Bash",
            input={
                "command": "pwd && ls -la",
                "description": "Show current directory and contents",
            },
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 2891,
        "cache_creation_input_tokens": 0,
        "cache_read_input_tokens": 68750,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 0,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 2,
    },
    message_id="msg_vrtx_012F3QXi1KiJ1Pqw9VUY29sX",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="e00e7f59-197c-451a-aa9b-3cdb151c5615",
)
# tool 结果回填是 UserMessage——不是用户打字，是 SDK 以"用户侧"身份把结果灌回对话
UserMessage(
    content=[
        ToolResultBlock(
            tool_use_id="toolu_vrtx_01R1SvMKn9DEfV2p6jZ8xFb6",
            content="/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk\ntotal 1024\ndrwxr-xr-x  29 liangzhu  staff     928 Jul  4 19:19 \x1b[1m\x1b[36m.\x1b[m\x1b[m\ndrwxr-xr-x  13 liangzhu  staff     416 Jul  4 19:19 \x1b[1m\x1b[36m..\x1b[m\x1b[m\n-rw-r--r--   1 liangzhu  staff    3957 Jul  4 17:29 00-总览与全局记忆图.ipynb\n-rw-r--r--   1 liangzhu  staff    1874 Jul  4 17:29 01-02-架构定位与底层原理.ipynb\n-rw-r--r--   1 liangzhu  staff    9366 Jul  4 17:55 03-安装与认证.ipynb\n-rw-r--r--   1 liangzhu  staff    6599 Jul  4 19:14 04-两个入口与两种输入模式.ipynb\n-rw-r--r--   1 liangzhu  staff   12744 Jul  4 19:19 05-最小可运行示例-query.ipynb\n-rw-r--r--   1 liangzhu  staff    9834 Jul  4 17:55 06-Agent-loop-工作原理.ipynb\n-rw-r--r--   1 liangzhu  staff   15718 Jul  4 17:56 07-核心技能-读懂消息流.ipynb\n-rw-r--r--   1 liangzhu  staff    5957 Jul  4 17:29 08-配置总线-ClaudeAgentOptions.ipynb\n-rw-r--r--   1 liangzhu  staff   15554 Jul  4 17:58 09-权限系统与Human-in-the-Loop.ipynb\n-rw-r--r--   1 liangzhu  staff    8242 Jul  4 17:59 10-自定义工具-进程内MCP.ipynb\n-rw-r--r--   1 liangzhu  staff    8642 Jul  4 17:53 11-Hooks-把agent变成可控系统.ipynb\n-rw-r--r--   1 liangzhu  staff    7804 Jul  4 17:59 12-外部MCP-接入现成生态.ipynb\n-rw-r--r--   1 liangzhu  staff   10824 Jul  4 17:41 13-会话-continue-resume-fork.ipynb\n-rw-r--r--   1 liangzhu  staff   51717 Jul  4 17:29 14-子Agent-把专项任务外包.ipynb\n-rw-r--r--   1 liangzhu  staff   11130 Jul  4 17:54 15-多轮与打断-ClaudeSDKClient.ipynb\n-rw-r--r--   1 liangzhu  staff    3637 Jul  4 17:52 16-流式输入-async-generator驱动会话.ipynb\n-rw-r--r--   1 liangzhu  staff    5043 Jul  4 17:51 17-流式输出-逐token拿增量.ipynb\n-rw-r--r--   1 liangzhu  staff    6463 Jul  4 17:52 18-结构化输出-验证过的JSON.ipynb\n-rw-r--r--   1 liangzhu  staff    4103 Jul  4 17:29 19-系统提示定制-四条路线.ipynb\n-rw-r--r--   1 liangzhu  staff   10318 Jul  4 17:40 20-文件系统特性-setting_sources总开关.ipynb\n-rw-r--r--   1 liangzhu  staff   17402 Jul  4 18:01 21-生产化-可运维的服务.ipynb\n-rw-r--r--   1 liangzhu  staff    1542 Jul  4 17:29 22-选型判断框架.ipynb\n-rw-r--r--   1 liangzhu  staff    3431 Jul  4 17:29 23-建议练习顺序.ipynb\n-rw-r--r--   1 liangzhu  staff  164850 Jul  3 18:30 Claude-Agent-SDK-Python-从0到1.ipynb\n-rw-r--r--   1 liangzhu  staff   16186 Jul  4 18:03 Claude-Agent-SDK-架构定位与子进程原理.md\n-rw-r--r--   1 liangzhu  staff    7312 Jul  4 17:29 _学习指南修订方案.md\n-rw-r--r--   1 liangzhu  staff   50277 Jul  4 17:09 test.ipynb",
            is_error=False,
        )
    ],
    uuid="df3ac006-a131-4d94-91ca-ea3dd1a7a2ec",
    parent_tool_use_id=None,
    tool_use_result={
        "stdout": "/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk\ntotal 1024\ndrwxr-xr-x  29 liangzhu  staff     928 Jul  4 19:19 \x1b[1m\x1b[36m.\x1b[m\x1b[m\ndrwxr-xr-x  13 liangzhu  staff     416 Jul  4 19:19 \x1b[1m\x1b[36m..\x1b[m\x1b[m\n-rw-r--r--   1 liangzhu  staff    3957 Jul  4 17:29 00-总览与全局记忆图.ipynb\n-rw-r--r--   1 liangzhu  staff    1874 Jul  4 17:29 01-02-架构定位与底层原理.ipynb\n-rw-r--r--   1 liangzhu  staff    9366 Jul  4 17:55 03-安装与认证.ipynb\n-rw-r--r--   1 liangzhu  staff    6599 Jul  4 19:14 04-两个入口与两种输入模式.ipynb\n-rw-r--r--   1 liangzhu  staff   12744 Jul  4 19:19 05-最小可运行示例-query.ipynb\n-rw-r--r--   1 liangzhu  staff    9834 Jul  4 17:55 06-Agent-loop-工作原理.ipynb\n-rw-r--r--   1 liangzhu  staff   15718 Jul  4 17:56 07-核心技能-读懂消息流.ipynb\n-rw-r--r--   1 liangzhu  staff    5957 Jul  4 17:29 08-配置总线-ClaudeAgentOptions.ipynb\n-rw-r--r--   1 liangzhu  staff   15554 Jul  4 17:58 09-权限系统与Human-in-the-Loop.ipynb\n-rw-r--r--   1 liangzhu  staff    8242 Jul  4 17:59 10-自定义工具-进程内MCP.ipynb\n-rw-r--r--   1 liangzhu  staff    8642 Jul  4 17:53 11-Hooks-把agent变成可控系统.ipynb\n-rw-r--r--   1 liangzhu  staff    7804 Jul  4 17:59 12-外部MCP-接入现成生态.ipynb\n-rw-r--r--   1 liangzhu  staff   10824 Jul  4 17:41 13-会话-continue-resume-fork.ipynb\n-rw-r--r--   1 liangzhu  staff   51717 Jul  4 17:29 14-子Agent-把专项任务外包.ipynb\n-rw-r--r--   1 liangzhu  staff   11130 Jul  4 17:54 15-多轮与打断-ClaudeSDKClient.ipynb\n-rw-r--r--   1 liangzhu  staff    3637 Jul  4 17:52 16-流式输入-async-generator驱动会话.ipynb\n-rw-r--r--   1 liangzhu  staff    5043 Jul  4 17:51 17-流式输出-逐token拿增量.ipynb\n-rw-r--r--   1 liangzhu  staff    6463 Jul  4 17:52 18-结构化输出-验证过的JSON.ipynb\n-rw-r--r--   1 liangzhu  staff    4103 Jul  4 17:29 19-系统提示定制-四条路线.ipynb\n-rw-r--r--   1 liangzhu  staff   10318 Jul  4 17:40 20-文件系统特性-setting_sources总开关.ipynb\n-rw-r--r--   1 liangzhu  staff   17402 Jul  4 18:01 21-生产化-可运维的服务.ipynb\n-rw-r--r--   1 liangzhu  staff    1542 Jul  4 17:29 22-选型判断框架.ipynb\n-rw-r--r--   1 liangzhu  staff    3431 Jul  4 17:29 23-建议练习顺序.ipynb\n-rw-r--r--   1 liangzhu  staff  164850 Jul  3 18:30 Claude-Agent-SDK-Python-从0到1.ipynb\n-rw-r--r--   1 liangzhu  staff   16186 Jul  4 18:03 Claude-Agent-SDK-架构定位与子进程原理.md\n-rw-r--r--   1 liangzhu  staff    7312 Jul  4 17:29 _学习指南修订方案.md\n-rw-r--r--   1 liangzhu  staff   50277 Jul  4 17:09 test.ipynb",
        "stderr": "",
        "interrupted": False,
        "isImage": False,
        "noOutputExpected": False,
    },
)
AssistantMessage(
    content=[
        ThinkingBlock(
            thinking="",
            signature="CAISuAMKZAgPEAIYAipAJC743nuvJdSosS1ynux6niU3lSEHn5Kivva2RNRn9+kVHBHb+a57Q/bTeNL4ZQJlAw/G/onnabNHJWzHHki6QDIOY2xhdWRlLWZhYmxlLTU4AUIIdGhpbmtpbmcSDIC8Siy4Q+yKfuMmMRoMjPie28EFTH6mTNadIjBMnXRWxQK/4E/XSJgrGJOCsf3arCKJ6ci/He7MI/3c9PQnCgIgkgo4dDgk3j/nG7QqgQKMfYLL+899z71u2NjsVZ3LtS+jXIbgrVUO76tBfiM3jw+Bfo5MiDEn/aJSPIwwq6LN/nyRWdwnwz+Muv6ayHrcbHBC1H6/SvGGgzdSSfauz4OUGXAJL9eQs1l8Tr/KBCcpP1QLQMJL6cwzCDcq0WIm0SkvDBsQcd/ffAyDl0wAeWGBGJWIJ+Kt8H5HAETvJaoxt4qrjyv1Q1J/iTDlX29j+ET4yf3E7F22z84XCW2zTVrBV1RBBfgerHLslnQg3QtIG4ZJ6kxAlLGh6jelRYJS+2hlw2yIAmAH4F07tm3O2GNsSIXSJU5+sNllbCrtzNp7B1W7VpB08XJeglhK57O09RgB",
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 2,
        "cache_creation_input_tokens": 142984,
        "cache_read_input_tokens": 0,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 142984,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 2,
    },
    message_id="msg_vrtx_01H9Yhn57SvhyqzKr8CGtA2B",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="e8dc536c-e1e5-4446-b12e-a6022cd1a5f4",
)
AssistantMessage(
    content=[
        ToolUseBlock(
            id="toolu_vrtx_01TMtL4qZ415sPs9hbErJfnM",
            name="Read",
            input={
                "file_path": "/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk/00-总览与全局记忆图.ipynb"
            },
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 2,
        "cache_creation_input_tokens": 142984,
        "cache_read_input_tokens": 0,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 142984,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 2,
    },
    message_id="msg_vrtx_01H9Yhn57SvhyqzKr8CGtA2B",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="fc39886b-e1bd-4a3d-bdc9-8bbfde1c8ebe",
)
UserMessage(
    content=[
        ToolResultBlock(
            tool_use_id="toolu_vrtx_01TMtL4qZ415sPs9hbErJfnM",
            content=[
                {
                    "text": '<cell id="d4d9b057"><cell_type>markdown</cell_type># Claude Agent SDK · Python 从 0 到 1\n\n> 来源：[Agent SDK overview](https://code.claude.com/docs/en/agent-sdk/overview)\n\n> 一句话本质：**Client SDK 让你"调用模型"，Agent SDK 让你"雇一个会自己用工具干活的 Claude"**。前者你得自己写 tool loop，后者循环内建。\n\n本笔记覆盖官方 Agent SDK 文档的全部主题（只讲 Python 版；TypeScript 仅在行为有差异时一句带过）。主线分四层递进：\n\n1. **跑起来**（§3–§7）：安装认证 → 两个入口 → agent loop 原理 → 读消息流。\n2. **控制它**（§8–§11）：配置总线 → 权限系统与 HITL → Hooks。\n3. **扩展它**（§10、§12–§14）：自定义工具 → 外部 MCP → 会话 → 子 agent。\n4. **上生产**（§15–§23）：多轮与打断 → 流式输入/输出 → 结构化输出 → 系统提示 → Claude Code 文件系统特性 → 生产化控制与安全部署。\n</cell id="d4d9b057">\n<cell id="1f31379e"><cell_type>markdown</cell_type>## 全局记忆图\n\n先把整张地图记住，后面每一节都是往这张图上填细节。主线是：**为什么 → 两个入口 → 一份配置 → 一条消息流 → 两个控制面 → 四类扩展 → 生产化**。\n\n```mermaid\nflowchart TD\n    A["为什么用 Agent SDK<br/>(不想手写 tool loop)"] --> B{选入口}\n    B -->|一次性任务| Q["query()<br/>发任务→收消息流"]\n    B -->|多轮/可打断| C["ClaudeSDKClient<br/>持续会话"]\n    Q --> O["ClaudeAgentOptions<br/>(唯一的配置总线)"]\n    C --> O\n    O --> S["消息流 async for<br/>System/Assistant/User/Result"]\n    S --> BLK["内容块<br/>Text / Thinking / ToolUse / ToolResult"]\n    S -.可选.-> SE["StreamEvent 逐 token 流式输出"]\n    S -.可选.-> SO["structured_output 结构化输出"]\n\n    O ==控制面==> P["权限系统<br/>hooks→deny→ask→mode→allow→回调"]\n    O ==控制面==> H["Hooks<br/>生命周期拦截/审计/改写"]\n\n    O -.扩展.-> T2["自定义工具<br/>@tool + 进程内 MCP"]\n    O -.扩展.-> T3["外部 MCP<br/>stdio / http / sse"]\n    O -.扩展.-> T4["会话<br/>continue / resume / fork"]\n    O -.扩展.-> T5["子 agent<br/>AgentDefinition"]\n    O -.扩展.-> T6["文件系统特性<br/>CLAUDE.md / skills / plugins"]\n\n    P & H --> PROD["生产化<br/>预算 / sandbox / 可观测 / 安全部署"]\n\n    style A fill:#fef3c7,stroke:#d97706\n    style O fill:#dbeafe,stroke:#2563eb\n    style S fill:#dcfce7,stroke:#16a34a\n    style BLK fill:#dcfce7,stroke:#16a34a\n    style P fill:#fee2e2,stroke:#dc2626\n    style H fill:#fee2e2,stroke:#dc2626\n    style PROD fill:#ede9fe,stroke:#7c3aed\n```\n\n四个颜色对应四个"必须先懂"的核心：黄=动机，蓝=配置总线（所有能力都从 `ClaudeAgentOptions` 挂进去），绿=消息流（agent 干活的过程全靠读这条流理解），红=控制面（权限与 hooks 决定 agent 能做什么）。\n</cell id="1f31379e">',
                    "type": "text",
                }
            ],
            is_error=None,
        )
    ],
    uuid="8232314f-030d-4705-b2e5-5cdbef7c4585",
    parent_tool_use_id=None,
    tool_use_result={
        "type": "notebook",
        "file": {
            "filePath": "/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk/00-总览与全局记忆图.ipynb",
            "cells": [
                {
                    "cellType": "markdown",
                    "source": '# Claude Agent SDK · Python 从 0 到 1\n\n> 来源：[Agent SDK overview](https://code.claude.com/docs/en/agent-sdk/overview)\n\n> 一句话本质：**Client SDK 让你"调用模型"，Agent SDK 让你"雇一个会自己用工具干活的 Claude"**。前者你得自己写 tool loop，后者循环内建。\n\n本笔记覆盖官方 Agent SDK 文档的全部主题（只讲 Python 版；TypeScript 仅在行为有差异时一句带过）。主线分四层递进：\n\n1. **跑起来**（§3–§7）：安装认证 → 两个入口 → agent loop 原理 → 读消息流。\n2. **控制它**（§8–§11）：配置总线 → 权限系统与 HITL → Hooks。\n3. **扩展它**（§10、§12–§14）：自定义工具 → 外部 MCP → 会话 → 子 agent。\n4. **上生产**（§15–§23）：多轮与打断 → 流式输入/输出 → 结构化输出 → 系统提示 → Claude Code 文件系统特性 → 生产化控制与安全部署。\n',
                    "cell_id": "d4d9b057",
                },
                {
                    "cellType": "markdown",
                    "source": '## 全局记忆图\n\n先把整张地图记住，后面每一节都是往这张图上填细节。主线是：**为什么 → 两个入口 → 一份配置 → 一条消息流 → 两个控制面 → 四类扩展 → 生产化**。\n\n```mermaid\nflowchart TD\n    A["为什么用 Agent SDK<br/>(不想手写 tool loop)"] --> B{选入口}\n    B -->|一次性任务| Q["query()<br/>发任务→收消息流"]\n    B -->|多轮/可打断| C["ClaudeSDKClient<br/>持续会话"]\n    Q --> O["ClaudeAgentOptions<br/>(唯一的配置总线)"]\n    C --> O\n    O --> S["消息流 async for<br/>System/Assistant/User/Result"]\n    S --> BLK["内容块<br/>Text / Thinking / ToolUse / ToolResult"]\n    S -.可选.-> SE["StreamEvent 逐 token 流式输出"]\n    S -.可选.-> SO["structured_output 结构化输出"]\n\n    O ==控制面==> P["权限系统<br/>hooks→deny→ask→mode→allow→回调"]\n    O ==控制面==> H["Hooks<br/>生命周期拦截/审计/改写"]\n\n    O -.扩展.-> T2["自定义工具<br/>@tool + 进程内 MCP"]\n    O -.扩展.-> T3["外部 MCP<br/>stdio / http / sse"]\n    O -.扩展.-> T4["会话<br/>continue / resume / fork"]\n    O -.扩展.-> T5["子 agent<br/>AgentDefinition"]\n    O -.扩展.-> T6["文件系统特性<br/>CLAUDE.md / skills / plugins"]\n\n    P & H --> PROD["生产化<br/>预算 / sandbox / 可观测 / 安全部署"]\n\n    style A fill:#fef3c7,stroke:#d97706\n    style O fill:#dbeafe,stroke:#2563eb\n    style S fill:#dcfce7,stroke:#16a34a\n    style BLK fill:#dcfce7,stroke:#16a34a\n    style P fill:#fee2e2,stroke:#dc2626\n    style H fill:#fee2e2,stroke:#dc2626\n    style PROD fill:#ede9fe,stroke:#7c3aed\n```\n\n四个颜色对应四个"必须先懂"的核心：黄=动机，蓝=配置总线（所有能力都从 `ClaudeAgentOptions` 挂进去），绿=消息流（agent 干活的过程全靠读这条流理解），红=控制面（权限与 hooks 决定 agent 能做什么）。\n',
                    "cell_id": "1f31379e",
                },
            ],
        },
    },
)
AssistantMessage(
    content=[
        ToolUseBlock(
            id="toolu_vrtx_017TfV75URspRC1vk5HVySeS",
            name="Read",
            input={
                "file_path": "/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk/_学习指南修订方案.md"
            },
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 2,
        "cache_creation_input_tokens": 142984,
        "cache_read_input_tokens": 0,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 142984,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 2,
    },
    message_id="msg_vrtx_01H9Yhn57SvhyqzKr8CGtA2B",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="8ae84667-0560-40ae-a119-cd82a21cea4d",
)
UserMessage(
    content=[
        ToolResultBlock(
            tool_use_id="toolu_vrtx_017TfV75URspRC1vk5HVySeS",
            content='1\t# 学习指南修订方案：真实例子驱动\n2\t\n3\t适用对象：本文件夹全部编号学习指南（00–23）。目标：消除"官方文档能看懂、学习指南反而难懂"的章节。\n4\t\n5\t## 病根诊断\n6\t\n7\t难懂章节的共同点：把官方文档用例子讲清的内容，重新抽象成规则叙述，读者被迫在脑中自行重建例子。典型症状：\n8\t\n9\t- 进例子之前先自造术语体系（"参与方有三个：……"）或铺场景剧本；\n10\t- 连续多段纯规则叙述，例子迟迟不出现；\n11\t- 例子只是规则的"附件"，而不是讲解的主线。\n12\t\n13\t对照：官方文档讲 resume 子 agent，结构是「一句话说明 → 两个 id 从哪来 → 三步操作 → 一段完整可运行代码 → 三条持久化规则」——例子和代码是主线，规则只做收束。\n14\t\n15\t## 每节写作模板（与官方文档结构对齐）\n16\t\n17\t0. **篇首标来源**——章标题正下方一行 `> 来源：[官方页标题](URL)`，多个页面用、分隔；URL 从官方 llms.txt 索引核对，不凭记忆写。\n18\t1. **一句话立目标**——这节解决什么问题，不超过两句，不铺场景剧本。\n19\t2. **真实例子做主线**——贴一次真实运行的消息/输出（裁剪到关键字段，保留真实 id 与真实值），逐段旁注"这是什么、从中取什么"。概念在例子中首次出现处就地引入，不提前集中定义。\n20\t3. **完整代码**——可运行的最小示例，中文注释指认关键行；优先改写官方示例。\n21\t4. **规则收束**——前提、边界条件用短 bullet 或表格放在最后，只收束例子已经展示过的内容。\n22\t\n23\t## 代码放哪：code cell vs markdown fenced block\n24\t\n25\tnotebook 的价值就在 cell 可执行，按可运行性分流：\n26\t\n27\t- **完整可运行示例**（import 齐全、无未定义名字、跑起来有意义）→ 必须放**独立 code cell**，读者一键执行。全放 markdown fenced block 的话，这个文件就不如直接写成 `.md`。\n28\t- **不可运行的示意** → 留在 markdown fenced block：真实消息实例的转录（`ResultMessage(...)` 之类）、依赖未定义名字的配置片段、协议骨架、目录树。\n29\t- 判别法：把这段代码原样贴进空 kernel 能不能跑？能跑 → code cell；不能 → markdown，且不要为了"看起来能跑"补假定义。\n30\t\n31\t## cell 粒度：一个章节一个 cell\n32\t\n33\t- 每个章节标题（`##` / `###`）独立成一个 markdown cell；一个 cell 不得跨两个章节。\n34\t- 章首 = `## N` 标题 + 导语合为一个 cell。\n35\t- 章节内插 code cell 时，code cell 之后的收束文字（如"三条边界"）单独成 markdown cell，仍属该章节。\n36\t- 批量拆分用脚本按标题行切（fenced block 内的 `#` 行不算标题），并断言拆分前后内容逐字拼接一致。脚本：\n37\t\n38\t```python\n39\t#!/usr/bin/env python3\n40\t"""把 notebook 的 markdown cell 按章节标题（##/###/####）拆成独立 cell。\n41\t\n42\t- code cell 原样保留（含输出）\n43\t- fenced code block 内的行不作为切分点\n44\t- 断言：每个 md cell 拆分后各段拼接 == 原文，保证零丢失\n45\t用法：python3 split_notebook_sections.py <a.ipynb> <b.ipynb> ...\n46\t"""\n47\timport json\n48\timport sys\n49\timport uuid\n50\tfrom pathlib import Path\n51\t\n52\t\n53\tdef split_markdown(lines):\n54\t    """按标题行切分，返回若干段（每段为行列表）。"""\n55\t    segments = []\n56\t    current = []\n57\t    in_fence = False\n58\t    for line in lines:\n59\t        stripped = line.lstrip()\n60\t        if stripped.startswith("```") or stripped.startswith("~~~"):\n61\t            in_fence = not in_fence\n62\t            current.append(line)\n63\t            continue\n64\t        is_heading = (not in_fence) and line.startswith("##") and line.lstrip("#").startswith(" ")\n65\t        if is_heading and any(l.strip() for l in current):\n66\t            segments.append(current)\n67\t            current = [line]\n68\t        else:\n69\t            current.append(line)\n70\t    if any(l.strip() for l in current):\n71\t        segments.append(current)\n72\t    return segments\n73\t\n74\t\n75\tdef to_lines(source):\n76\t    if isinstance(source, str):\n77\t        return source.splitlines(keepends=True)\n78\t    return list(source)\n79\t\n80\t\n81\tdef process(path):\n82\t    nb = json.loads(Path(path).read_text())\n83\t    new_cells = []\n84\t    changed = False\n85\t    for cell in nb["cells"]:\n86\t        if cell["cell_type"] != "markdown":\n87\t            new_cells.append(cell)\n88\t            continue\n89\t        lines = to_lines(cell["source"])\n90\t        segments = split_markdown(lines)\n91\t        assert "".join(l for seg in segments for l in seg) == "".join(lines), f"{path}: 内容不一致"\n92\t        if len(segments) <= 1:\n93\t            new_cells.append(cell)\n94\t            continue\n95\t        changed = True\n96\t        for i, seg in enumerate(segments):\n97\t            while seg and not seg[-1].strip():\n98\t                seg.pop()\n99\t            new_cells.append({\n100\t                "cell_type": "markdown",\n101\t                "id": cell["id"] if i == 0 else uuid.uuid4().hex[:8],\n102\t                "metadata": dict(cell.get("metadata", {})),\n103\t                "source": seg,\n104\t            })\n105\t    if changed:\n106\t        nb["cells"] = new_cells\n107\t        Path(path).write_text(json.dumps(nb, ensure_ascii=False, indent=1) + "\\n")\n108\t    print(f"{\'SPLIT \' if changed else \'keep  \'}{Path(path).name}")\n109\t\n110\t\n111\tif __name__ == "__main__":\n112\t    for p in sys.argv[1:]:\n113\t        process(p)\n114\t```\n115\t\n116\t## 真实实例的来源（按优先级）\n117\t\n118\t1. 本 notebook 已执行 cell 的真实输出；\n119\t2. 真实会话采集的消息流记录（如用户提供的运行日志）；\n120\t3. 官方文档中的示例（视为合格锚点）；\n121\t4. 都没有 → 写一个"采集 cell"（运行后打印原始消息流），标注待运行；**禁止虚构 id/值冒充真实数据**。\n122\t\n123\t## 禁止项\n124\t\n125\t- 自造术语体系开场，再让读者带着术语进例子；\n126\t- 连续三段以上无代码、无实例的规则叙述；\n127\t- 官方文档本来例子驱动的章节，改写成纯规则版；\n128\t- 假 id、占位值伪装成真实运行数据（官方示例中的示意值除外，但要能区分）。\n129\t\n130\t## 自检法\n131\t\n132\t逐节问四个问题：\n133\t\n134\t1. 官方文档对应章节怎么讲的？它用了例子，这一节的例子在哪？\n135\t2. **官方页的每一段示例代码，指南里是否都有对应的 code cell？**把官方代码"化成叙述"即违规——对计算机行为最清晰的说明就是代码本身。修订前先列官方页的代码示例清单，逐条勾对。\n136\t3. 删掉所有例子和代码后这节还剩几段？剩三段以上说明规则堆太多；\n137\t4. 例子里的值是真的吗？能指出它来自哪次运行、哪篇官方文档吗？\n138\t\n139\t## 批量修订流程\n140\t\n141\t1. 逐文件审计，按上面标准列出违规小节（文件、cell、小节标题、违规类型）；\n142\t2. 每个违规小节按模板重写：优先复用该 notebook 已有的执行输出做锚点；\n143\t3. 一次只改一个文件，改完确认 notebook JSON 结构完整（cell 顺序、id 不乱）；\n144\t4. VSCode 中打开着的 notebook 先关闭再改，避免保存竞态；乱序后整本重建，不逐 cell 修补。\n145\t\n146\t## 已按此方案修订的范例\n147\t\n148\t`14-子Agent-把专项任务外包.ipynb` 的 §14.2.2（跨重启 resume）：以一次真实 poem 派发的消息流为主线（真实 agentId `abbacb99b2fc86fc3`、真实 session_id），两个 id 的位置直接贴真实消息，随后三步操作 + 完整双 `query()` 代码 + 三条边界收束。\n149\t',
            is_error=None,
        )
    ],
    uuid="6e7a3768-134f-4180-bbf0-c74856da912a",
    parent_tool_use_id=None,
    tool_use_result={
        "type": "text",
        "file": {
            "filePath": "/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk/_学习指南修订方案.md",
            "content": '# 学习指南修订方案：真实例子驱动\n\n适用对象：本文件夹全部编号学习指南（00–23）。目标：消除"官方文档能看懂、学习指南反而难懂"的章节。\n\n## 病根诊断\n\n难懂章节的共同点：把官方文档用例子讲清的内容，重新抽象成规则叙述，读者被迫在脑中自行重建例子。典型症状：\n\n- 进例子之前先自造术语体系（"参与方有三个：……"）或铺场景剧本；\n- 连续多段纯规则叙述，例子迟迟不出现；\n- 例子只是规则的"附件"，而不是讲解的主线。\n\n对照：官方文档讲 resume 子 agent，结构是「一句话说明 → 两个 id 从哪来 → 三步操作 → 一段完整可运行代码 → 三条持久化规则」——例子和代码是主线，规则只做收束。\n\n## 每节写作模板（与官方文档结构对齐）\n\n0. **篇首标来源**——章标题正下方一行 `> 来源：[官方页标题](URL)`，多个页面用、分隔；URL 从官方 llms.txt 索引核对，不凭记忆写。\n1. **一句话立目标**——这节解决什么问题，不超过两句，不铺场景剧本。\n2. **真实例子做主线**——贴一次真实运行的消息/输出（裁剪到关键字段，保留真实 id 与真实值），逐段旁注"这是什么、从中取什么"。概念在例子中首次出现处就地引入，不提前集中定义。\n3. **完整代码**——可运行的最小示例，中文注释指认关键行；优先改写官方示例。\n4. **规则收束**——前提、边界条件用短 bullet 或表格放在最后，只收束例子已经展示过的内容。\n\n## 代码放哪：code cell vs markdown fenced block\n\nnotebook 的价值就在 cell 可执行，按可运行性分流：\n\n- **完整可运行示例**（import 齐全、无未定义名字、跑起来有意义）→ 必须放**独立 code cell**，读者一键执行。全放 markdown fenced block 的话，这个文件就不如直接写成 `.md`。\n- **不可运行的示意** → 留在 markdown fenced block：真实消息实例的转录（`ResultMessage(...)` 之类）、依赖未定义名字的配置片段、协议骨架、目录树。\n- 判别法：把这段代码原样贴进空 kernel 能不能跑？能跑 → code cell；不能 → markdown，且不要为了"看起来能跑"补假定义。\n\n## cell 粒度：一个章节一个 cell\n\n- 每个章节标题（`##` / `###`）独立成一个 markdown cell；一个 cell 不得跨两个章节。\n- 章首 = `## N` 标题 + 导语合为一个 cell。\n- 章节内插 code cell 时，code cell 之后的收束文字（如"三条边界"）单独成 markdown cell，仍属该章节。\n- 批量拆分用脚本按标题行切（fenced block 内的 `#` 行不算标题），并断言拆分前后内容逐字拼接一致。脚本：\n\n```python\n#!/usr/bin/env python3\n"""把 notebook 的 markdown cell 按章节标题（##/###/####）拆成独立 cell。\n\n- code cell 原样保留（含输出）\n- fenced code block 内的行不作为切分点\n- 断言：每个 md cell 拆分后各段拼接 == 原文，保证零丢失\n用法：python3 split_notebook_sections.py <a.ipynb> <b.ipynb> ...\n"""\nimport json\nimport sys\nimport uuid\nfrom pathlib import Path\n\n\ndef split_markdown(lines):\n    """按标题行切分，返回若干段（每段为行列表）。"""\n    segments = []\n    current = []\n    in_fence = False\n    for line in lines:\n        stripped = line.lstrip()\n        if stripped.startswith("```") or stripped.startswith("~~~"):\n            in_fence = not in_fence\n            current.append(line)\n            continue\n        is_heading = (not in_fence) and line.startswith("##") and line.lstrip("#").startswith(" ")\n        if is_heading and any(l.strip() for l in current):\n            segments.append(current)\n            current = [line]\n        else:\n            current.append(line)\n    if any(l.strip() for l in current):\n        segments.append(current)\n    return segments\n\n\ndef to_lines(source):\n    if isinstance(source, str):\n        return source.splitlines(keepends=True)\n    return list(source)\n\n\ndef process(path):\n    nb = json.loads(Path(path).read_text())\n    new_cells = []\n    changed = False\n    for cell in nb["cells"]:\n        if cell["cell_type"] != "markdown":\n            new_cells.append(cell)\n            continue\n        lines = to_lines(cell["source"])\n        segments = split_markdown(lines)\n        assert "".join(l for seg in segments for l in seg) == "".join(lines), f"{path}: 内容不一致"\n        if len(segments) <= 1:\n            new_cells.append(cell)\n            continue\n        changed = True\n        for i, seg in enumerate(segments):\n            while seg and not seg[-1].strip():\n                seg.pop()\n            new_cells.append({\n                "cell_type": "markdown",\n                "id": cell["id"] if i == 0 else uuid.uuid4().hex[:8],\n                "metadata": dict(cell.get("metadata", {})),\n                "source": seg,\n            })\n    if changed:\n        nb["cells"] = new_cells\n        Path(path).write_text(json.dumps(nb, ensure_ascii=False, indent=1) + "\\n")\n    print(f"{\'SPLIT \' if changed else \'keep  \'}{Path(path).name}")\n\n\nif __name__ == "__main__":\n    for p in sys.argv[1:]:\n        process(p)\n```\n\n## 真实实例的来源（按优先级）\n\n1. 本 notebook 已执行 cell 的真实输出；\n2. 真实会话采集的消息流记录（如用户提供的运行日志）；\n3. 官方文档中的示例（视为合格锚点）；\n4. 都没有 → 写一个"采集 cell"（运行后打印原始消息流），标注待运行；**禁止虚构 id/值冒充真实数据**。\n\n## 禁止项\n\n- 自造术语体系开场，再让读者带着术语进例子；\n- 连续三段以上无代码、无实例的规则叙述；\n- 官方文档本来例子驱动的章节，改写成纯规则版；\n- 假 id、占位值伪装成真实运行数据（官方示例中的示意值除外，但要能区分）。\n\n## 自检法\n\n逐节问四个问题：\n\n1. 官方文档对应章节怎么讲的？它用了例子，这一节的例子在哪？\n2. **官方页的每一段示例代码，指南里是否都有对应的 code cell？**把官方代码"化成叙述"即违规——对计算机行为最清晰的说明就是代码本身。修订前先列官方页的代码示例清单，逐条勾对。\n3. 删掉所有例子和代码后这节还剩几段？剩三段以上说明规则堆太多；\n4. 例子里的值是真的吗？能指出它来自哪次运行、哪篇官方文档吗？\n\n## 批量修订流程\n\n1. 逐文件审计，按上面标准列出违规小节（文件、cell、小节标题、违规类型）；\n2. 每个违规小节按模板重写：优先复用该 notebook 已有的执行输出做锚点；\n3. 一次只改一个文件，改完确认 notebook JSON 结构完整（cell 顺序、id 不乱）；\n4. VSCode 中打开着的 notebook 先关闭再改，避免保存竞态；乱序后整本重建，不逐 cell 修补。\n\n## 已按此方案修订的范例\n\n`14-子Agent-把专项任务外包.ipynb` 的 §14.2.2（跨重启 resume）：以一次真实 poem 派发的消息流为主线（真实 agentId `abbacb99b2fc86fc3`、真实 session_id），两个 id 的位置直接贴真实消息，随后三步操作 + 完整双 `query()` 代码 + 三条边界收束。\n',
            "numLines": 149,
            "startLine": 1,
            "totalLines": 149,
        },
    },
)
AssistantMessage(
    content=[
        ThinkingBlock(
            thinking="",
            signature="CAISgwQKZAgPEAIYAipALbqnktQO9o0++r2yfmap+ePqXk+TXrxs++4qbhcjzfcpqNSIv1VilCaLWnKgbz95bdIA+ZNIaI5bEpGDjoNUZDIOY2xhdWRlLWZhYmxlLTU4AUIIdGhpbmtpbmcSDPq86dD7MRaSZCA2QhoMaS42hRtXEt911zIQIjBS8gkyLdPKDMz8+696YSNeQWX3R07SKZSTbUlnKFKikVQBMRY5NOYB8WspVvQimjwqzAIz/y+R2jPtjRD3nAZJ/0UcmIE7TwEorHOHjCVG3F0zfUx9/zc0f7UI4+XxCS5rPpnuWq0pymW4GdZD8lUseeEMUMl+P8btX7DdTuYoR0kzyI6guBJWJV9X0XxiyY+L26AOvHvQZBq3EC3gRk9j0bT3iKe23WOb1OUe5ZlxlndlfGGaA+zLYqNGAzz/CHM3+Wg/0Hdfjd8kgJFIc+ewYcwAh1wALd9JtyqNmskpfLVRcZrr7r88mFaDCrv5uc72ZoQVMNCJtdIqXx1q9tOuigHdtO5ZtzReZQqTQVMXDZwy15RlRG152Sgw0GiwNYeVFniuQcyyr7xs/YYMpDzZKSdBbnTeIq34H19N8NBrLe6w4NnyhS/BLLOCAkAKg7bo9CAP01Z6lEi/05hsJM5KwXKMP3KSJkcLrFd2rbaQNuo0eP9yFtmS/fB3+xJosxgB",
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 1834,
        "cache_creation_input_tokens": 4960,
        "cache_read_input_tokens": 142984,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 4960,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 6,
    },
    message_id="msg_vrtx_01QsAsh5zWfED4DMEMvRh3ne",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="dd1795ec-b5e4-47b8-b9e9-febf3fc72c7e",
)
AssistantMessage(
    content=[
        TextBlock(
            text="这是一个**学习笔记项目**，不是代码工程：Obsidian vault 中关于 **Claude Agent SDK（Python 版）** 的系列教学 notebook，位于 `study-notes/AI/agents/claude-agent-sdk/`。\n\n## 项目结构\n\n由 24 个编号 notebook（00–23）组成，按四层递进主线组织（`00-总览与全局记忆图.ipynb` 是全书地图）：\n\n**第一层：跑起来（§1–§7）**\n- `01-02` 架构定位与底层原理（SDK 通过子进程驱动 Claude Code CLI）\n- `03` 安装与认证\n- `04` 两个入口（`query()` 一次性任务 vs `ClaudeSDKClient` 持续会话）与两种输入模式\n- `05` 最小可运行示例\n- `06` Agent loop 工作原理\n- `07` 读懂消息流（System/Assistant/User/Result 四类消息 + Text/Thinking/ToolUse/ToolResult 内容块）\n\n**第二层：控制它（§8–§11）**\n- `08` 配置总线 `ClaudeAgentOptions` —— 所有能力的唯一挂载点\n- `09` 权限系统与 Human-in-the-Loop（hooks→deny→ask→mode→allow→回调 的判定链）\n- `11` Hooks —— 生命周期拦截/审计/改写\n\n**第三层：扩展它（§10、§12–§14）**\n- `10` 自定义工具（`@tool` + 进程内 MCP）\n- `12` 外部 MCP（stdio/http/sse）\n- `13` 会话管理（continue/resume/fork）\n- `14` 子 Agent（`AgentDefinition`，本目录最大的一篇，51KB）\n\n**第四层：上生产（§15–§23）**\n- `15` 多轮与打断、`16` 流式输入、`17` 流式输出（逐 token）、`18` 结构化输出\n- `19` 系统提示定制、`20` 文件系统特性（`setting_sources`、CLAUDE.md/skills/plugins）\n- `21` 生产化（预算/sandbox/可观测/安全部署）、`22` 选型判断框架、`23` 练习顺序\n\n## 支撑文件\n\n- `_学习指南修订方案.md` —— 全套笔记的写作规范：真实例子驱动、可运行代码放独立 code cell、一章节一 cell，附批量拆分脚本\n- `Claude-Agent-SDK-Python-从0到1.ipynb`（165KB）—— 拆分前的旧版单体笔记\n- `Claude-Agent-SDK-架构定位与子进程原理.md`、`test.ipynb` —— 补充材料与试验文件\n\n一句话概括：这套笔记覆盖官方 Agent SDK 文档全部主题，主线是「为什么 → 两个入口 → 一份配置总线 → 一条消息流 → 两个控制面（权限+Hooks）→ 四类扩展 → 生产化」。"
        )
    ],
    model="claude-fable-5",
    parent_tool_use_id=None,
    error=None,
    usage={
        "input_tokens": 1834,
        "cache_creation_input_tokens": 4960,
        "cache_read_input_tokens": 142984,
        "cache_creation": {
            "ephemeral_5m_input_tokens": 4960,
            "ephemeral_1h_input_tokens": 0,
        },
        "output_tokens": 6,
    },
    message_id="msg_vrtx_01QsAsh5zWfED4DMEMvRh3ne",
    stop_reason=None,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    uuid="8dca33da-77be-45fa-822e-65caa2e693b8",
)
# 收尾是 ResultMessage（完整实例见 §7.3）
ResultMessage(
    subtype="success",
    duration_ms=31301,
    duration_api_ms=30094,
    is_error=False,
    num_turns=4,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    stop_reason="end_turn",
    total_cost_usd=2.187504,
    usage={
        "input_tokens": 4727,
        "cache_creation_input_tokens": 147944,
        "cache_read_input_tokens": 211734,
        "output_tokens": 1584,
        "server_tool_use": {"web_search_requests": 0, "web_fetch_requests": 0},
        "service_tier": "standard",
        "cache_creation": {
            "ephemeral_1h_input_tokens": 0,
            "ephemeral_5m_input_tokens": 147944,
        },
        "inference_geo": "",
        "iterations": [],
        "speed": "standard",
    },
    result="这是一个**学习笔记项目**，不是代码工程：Obsidian vault 中关于 **Claude Agent SDK（Python 版）** 的系列教学 notebook，位于 `study-notes/AI/agents/claude-agent-sdk/`。\n\n## 项目结构\n\n由 24 个编号 notebook（00–23）组成，按四层递进主线组织（`00-总览与全局记忆图.ipynb` 是全书地图）：\n\n**第一层：跑起来（§1–§7）**\n- `01-02` 架构定位与底层原理（SDK 通过子进程驱动 Claude Code CLI）\n- `03` 安装与认证\n- `04` 两个入口（`query()` 一次性任务 vs `ClaudeSDKClient` 持续会话）与两种输入模式\n- `05` 最小可运行示例\n- `06` Agent loop 工作原理\n- `07` 读懂消息流（System/Assistant/User/Result 四类消息 + Text/Thinking/ToolUse/ToolResult 内容块）\n\n**第二层：控制它（§8–§11）**\n- `08` 配置总线 `ClaudeAgentOptions` —— 所有能力的唯一挂载点\n- `09` 权限系统与 Human-in-the-Loop（hooks→deny→ask→mode→allow→回调 的判定链）\n- `11` Hooks —— 生命周期拦截/审计/改写\n\n**第三层：扩展它（§10、§12–§14）**\n- `10` 自定义工具（`@tool` + 进程内 MCP）\n- `12` 外部 MCP（stdio/http/sse）\n- `13` 会话管理（continue/resume/fork）\n- `14` 子 Agent（`AgentDefinition`，本目录最大的一篇，51KB）\n\n**第四层：上生产（§15–§23）**\n- `15` 多轮与打断、`16` 流式输入、`17` 流式输出（逐 token）、`18` 结构化输出\n- `19` 系统提示定制、`20` 文件系统特性（`setting_sources`、CLAUDE.md/skills/plugins）\n- `21` 生产化（预算/sandbox/可观测/安全部署）、`22` 选型判断框架、`23` 练习顺序\n\n## 支撑文件\n\n- `_学习指南修订方案.md` —— 全套笔记的写作规范：真实例子驱动、可运行代码放独立 code cell、一章节一 cell，附批量拆分脚本\n- `Claude-Agent-SDK-Python-从0到1.ipynb`（165KB）—— 拆分前的旧版单体笔记\n- `Claude-Agent-SDK-架构定位与子进程原理.md`、`test.ipynb` —— 补充材料与试验文件\n\n一句话概括：这套笔记覆盖官方 Agent SDK 文档全部主题，主线是「为什么 → 两个入口 → 一份配置总线 → 一条消息流 → 两个控制面（权限+Hooks）→ 四类扩展 → 生产化」。",
    structured_output=None,
    model_usage={
        "claude-fable-5": {
            "inputTokens": 4727,
            "outputTokens": 1584,
            "cacheReadInputTokens": 211734,
            "cacheCreationInputTokens": 147944,
            "webSearchRequests": 0,
            "costUSD": 2.187504,
            "contextWindow": 200000,
            "maxOutputTokens": 64000,
        }
    },
    permission_denials=[],
    deferred_tool_use=None,
    errors=None,
    api_error_status=None,
    uuid="ac089b40-9cf5-4675-a6e4-48bf330b60b9",
)



六种主类型全表：

| 类型 | 含义 | 关键字段 |
|---|---|---|
| `SystemMessage` | 系统事件（含一组具名子类，见 §7.2「SystemMessage 家族」） | `subtype`、`data` |
| `AssistantMessage` | Claude 的一轮输出 | `content`（一串 ContentBlock）、`model`、`parent_tool_use_id`、`error`、`usage`、`message_id` |
| `UserMessage` | 用户输入 / tool 结果回填 | `content`、`parent_tool_use_id`、`tool_use_result`、`uuid`（file checkpointing 的 checkpoint ID，见 §21.4「文件快照与任务清单」） |
| `StreamEvent` | 逐 token 增量事件（仅开启 `include_partial_messages` 时出现，本次运行未开启所以流里没有） | `event`（原始 API 事件 dict），见 §17「流式输出」 |
| `RateLimitEvent` | 限流状态变化时推送 | `rate_limit_info`：`status`（`allowed` / `allowed_warning` / `rejected`）、`resets_at`、`utilization`、`rate_limit_type`（另有 `overage_*` 系列与 `raw`） |
| `ResultMessage` | 本次任务**收尾** | `result`、`subtype`、`is_error`、`num_turns`、`duration_ms` / `duration_api_ms`、`session_id`、`total_cost_usd`、`usage`、`model_usage`、`structured_output`、`stop_reason`、`permission_denials`、`api_error_status`、`errors`、`deferred_tool_use`、`uuid` |

- `AssistantMessage.error` 是本轮失败的分类标签，七个取值：`authentication_failed` / `billing_error` / `rate_limit` / `invalid_request` / `server_error` / `max_output_tokens` / `unknown`。
- `RateLimitEvent` 用于限流预警：`allowed_warning` 时提示用户接近限额，`rejected` 时主动退避到 `resets_at`。


### 7.2 SystemMessage 家族：通用 subtype + 六个具名子类

通用 `SystemMessage` 的常见 `subtype`：`"init"`（首条，见上方真实实例）、`"compact_boundary"`（发生了 compaction）、`"informational"`（状态横幅）、`"worker_shutting_down"`（宿主进程要退出或 Remote Control 断开，当前 turn 结束后循环将终止）。

六个高频 subtype 有具名子类，字段直接是属性、不用翻 `data` dict。


全表：

| 子类 | 何时出现 | 关键字段 |
|---|---|---|
| `HookEventMessage` | hook 开始/结束（需 `include_hook_events=True`；subtype 为 `hook_started` / `hook_response`） | `hook_event_name`（如 `"PreToolUse"`）；`hook_response` 的 `data` 里有 `output` / `exit_code` / `outcome` |
| `TaskStartedMessage` | 后台任务启动 | `task_id`、`description`、`task_type`、`tool_use_id` |
| `TaskProgressMessage` | 后台任务运行中定期上报 | `task_id`、`usage`（`total_tokens` / `tool_uses` / `duration_ms`）、`last_tool_name` |
| `TaskNotificationMessage` | 后台任务完成 / 失败 / 被停止 | `task_id`、`status`（`completed` / `failed` / `stopped`）、`summary`、`output_file`、`usage` |
| `TaskUpdatedMessage` | 后台任务状态变更 | `task_id`、`patch`（变更字段）、`status`（`pending` / `running` / `paused` / `completed` / `failed` / `killed`） |
| `MirrorErrorMessage` | `session_store` 镜像写失败（SDK 合成，见 §13.3「session_store」） | `key`、`error` |

"任务（task）"指主对话之外**后台并行运行的工作**：主消息流继续往前走，任务在旁边跑，靠上表的 Task 系消息汇报进展。`task_type` 标注任务类别：

- `local_agent` —— 后台派发的子 agent（下方真实序列即是）；
- `local_bash` —— 后台运行的 Bash 命令（`run_in_background=True` 启动的命令、`Monitor` 监视的脚本）；
- `remote_agent` —— 在远端云环境运行的 agent。

每个任务有唯一 `task_id`，`client.stop_task(task_id)` 可中途停止。

两个使用要点：

- **所有子类都继承 `SystemMessage`**。
- **任务结束不保证发 `TaskNotificationMessage`，判断"任务结束"必须同时盯 `TaskUpdatedMessage`**。status 分两类：`pending` / `running` / `paused` 是中间态（之后还会变），`completed` / `failed` / `stopped` / `killed` 是终态（进入后不再变化，任务到此为止）。`stop_task()` 强杀的任务往往只发一条 `TaskUpdatedMessage`（`patch` 里 `status` 变为 `"killed"`），之后再无任何消息——只等 notification 的代码会让它在活跃任务表里变成永远删不掉的僵尸；正常完成的任务则两条都发（下方真实序列即是：update 先到、notification 后到）。落地做法：收到任一种消息、其 status 落入终态集合就移除该 `task_id`（同一任务可能触发两次移除，处理需幂等）。终态集合 SDK 已定义好，可直接 `from claude_agent_sdk import TERMINAL_TASK_STATUSES`。

In [ ]:
# 一次真实会话（Claude Code 环境下派发 `poem-style` 子 agent）里它们长这样，按实际到达顺序：
HookEventMessage(subtype='hook_started', hook_event_name='SessionStart', data={'hook_id': '1ae3e4ee-755a-4b98-877a-1136b41dc765', 'hook_name': 'SessionStart:startup', ...})
HookEventMessage(subtype='hook_response', hook_event_name='SessionStart', data={..., 'output': '', 'exit_code': 0, 'outcome': 'success'})

# 后台任务（这里是后台派发的子 agent）的生命周期三连
TaskStartedMessage(task_id='abbacb99b2fc86fc3', description='Get poem style guidance', task_type='local_agent', tool_use_id='toolu_vrtx_017X31vc94xcctmXnKC6sqrK')
TaskUpdatedMessage(task_id='abbacb99b2fc86fc3', status='completed', patch={'status': 'completed', 'end_time': 1783153946754})
TaskNotificationMessage(task_id='abbacb99b2fc86fc3', status='completed', summary='Get poem style guidance', usage={'total_tokens': 13230, 'tool_uses': 0, 'duration_ms': 20283})

这次任务 20 秒就完成，所以没出现 `TaskProgressMessage`（运行中定期上报才有）；`MirrorErrorMessage` 只在 `session_store` 镜像写失败时由 SDK 合成（见 §13.3「session_store」）。

### 7.3 ResultMessage 的 subtype 全表

一条真实的 `success` 收尾（上面那次 poem 会话，长字段截断）：


In [ ]:
ResultMessage(
    subtype="success",
    duration_ms=31301,
    duration_api_ms=30094,
    is_error=False,
    num_turns=4,
    session_id="d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb",
    stop_reason="end_turn",
    total_cost_usd=2.187504,
    usage={
        "input_tokens": 4727,
        "cache_creation_input_tokens": 147944,
        "cache_read_input_tokens": 211734,
        "output_tokens": 1584,
        "server_tool_use": {"web_search_requests": 0, "web_fetch_requests": 0},
        "service_tier": "standard",
        "cache_creation": {
            "ephemeral_1h_input_tokens": 0,
            "ephemeral_5m_input_tokens": 147944,
        },
        "inference_geo": "",
        "iterations": [],
        "speed": "standard",
    },
    result="这是一个**学习笔记项目**，不是代码工程：Obsidian vault 中关于 **Claude Agent SDK（Python 版）** 的系列教学 notebook，位于 `study-notes/AI/agents/claude-agent-sdk/`。\n\n## 项目结构\n\n由 24 个编号 notebook（00–23）组成，按四层递进主线组织（`00-总览与全局记忆图.ipynb` 是全书地图）：\n\n**第一层：跑起来（§1–§7）**\n- `01-02` 架构定位与底层原理（SDK 通过子进程驱动 Claude Code CLI）\n- `03` 安装与认证\n- `04` 两个入口（`query()` 一次性任务 vs `ClaudeSDKClient` 持续会话）与两种输入模式\n- `05` 最小可运行示例\n- `06` Agent loop 工作原理\n- `07` 读懂消息流（System/Assistant/User/Result 四类消息 + Text/Thinking/ToolUse/ToolResult 内容块）\n\n**第二层：控制它（§8–§11）**\n- `08` 配置总线 `ClaudeAgentOptions` —— 所有能力的唯一挂载点\n- `09` 权限系统与 Human-in-the-Loop（hooks→deny→ask→mode→allow→回调 的判定链）\n- `11` Hooks —— 生命周期拦截/审计/改写\n\n**第三层：扩展它（§10、§12–§14）**\n- `10` 自定义工具（`@tool` + 进程内 MCP）\n- `12` 外部 MCP（stdio/http/sse）\n- `13` 会话管理（continue/resume/fork）\n- `14` 子 Agent（`AgentDefinition`，本目录最大的一篇，51KB）\n\n**第四层：上生产（§15–§23）**\n- `15` 多轮与打断、`16` 流式输入、`17` 流式输出（逐 token）、`18` 结构化输出\n- `19` 系统提示定制、`20` 文件系统特性（`setting_sources`、CLAUDE.md/skills/plugins）\n- `21` 生产化（预算/sandbox/可观测/安全部署）、`22` 选型判断框架、`23` 练习顺序\n\n## 支撑文件\n\n- `_学习指南修订方案.md` —— 全套笔记的写作规范：真实例子驱动、可运行代码放独立 code cell、一章节一 cell，附批量拆分脚本\n- `Claude-Agent-SDK-Python-从0到1.ipynb`（165KB）—— 拆分前的旧版单体笔记\n- `Claude-Agent-SDK-架构定位与子进程原理.md`、`test.ipynb` —— 补充材料与试验文件\n\n一句话概括：这套笔记覆盖官方 Agent SDK 文档全部主题，主线是「为什么 → 两个入口 → 一份配置总线 → 一条消息流 → 两个控制面（权限+Hooks）→ 四类扩展 → 生产化」。",
    structured_output=None,
    model_usage={
        "claude-fable-5": {
            "inputTokens": 4727,
            "outputTokens": 1584,
            "cacheReadInputTokens": 211734,
            "cacheCreationInputTokens": 147944,
            "webSearchRequests": 0,
            "costUSD": 2.187504,
            "contextWindow": 200000,
            "maxOutputTokens": 64000,
        }
    },
    permission_denials=[],
    deferred_tool_use=None,
    errors=None,
    api_error_status=None,
    uuid="ac089b40-9cf5-4675-a6e4-48bf330b60b9",
)

subtype 决定 `result` 能不能读：

| subtype | 含义 | `result` 是否可用 |
|---|---|---|
| `success` | 正常完成 | ✅ |
| `error_max_turns` | 触到 `max_turns` | ❌ |
| `error_max_budget_usd` | 触到 `max_budget_usd` | ❌ |
| `error_during_execution` | 执行中断（API 失败、请求取消等） | ❌ |
| `error_max_structured_output_retries` | 结构化输出重试耗尽（见 §18） | ❌ |

五个配套的坑：

- **`result` 只在 `success` 上存在**，读之前必须先查 subtype；所有 subtype 都带 `session_id`（错误后也能 resume）。
- 一个例外形态：`subtype="success"` 但 `is_error=True` 时，`result` 装的是 API 错误串（可能为空）——消费 `result` 前把 `is_error` 也看一眼。`errors` 字段则只在 `error_*` subtype 上填充。
- `total_cost_usd` / `usage` 类型是 optional，某些错误路径上是 `None`，格式化前先判空。
- **`ResultMessage` 不保证是流的最后一条消息**。它是"本轮结果"，但之后可能还尾随零星消息（如 prompt 建议事件），一收到 result 就 `break` 会漏掉它们——让 `async for` 迭代到流自然耗尽再退出。
- `api_error_status` 记录 API 失败的 HTTP 状态码（429 / 500 / 529），且**只在 `subtype="success"` 时填充**——含义是"过程中撞过 API 错误、被重试吸收后最终成功"，可直接进日志和告警（上面实例里是 `None`——全程没撞过）。

`stop_reason`（`str | None`）记录模型最后一轮为何停止：`end_turn`（正常，见上方实例）、`max_tokens`、`refusal`（可用它检测模型拒绝）。

### 7.4 ContentBlock 四种块（在 `AssistantMessage.content` 里）

§7.1「Message 六种主类型」 的真实节选里已出现三种：`TextBlock`（"I'll explore the project structure..."）、`ToolUseBlock`（`name='Bash'`）、`ToolResultBlock`（`content='/Users/liangzhu/...'`）。第四种 `ThinkingBlock` 在 poem 会话里长这样：

```python
# 真实流里 thinking 可能是空串、只带 signature——加密思考内容不回传时就这样，
# 渲染代码不要假设 .thinking 一定有文字
ThinkingBlock(thinking='', signature='CAISrgMKZAgPEAIYAipA0kDjhsBlaBAExriAaM8u...')
```

| 块 | 含义 |
|---|---|
| `TextBlock` | 普通文本（`.text`） |
| `ThinkingBlock` | 扩展思考内容（`.thinking`、`.signature`） |
| `ToolUseBlock` | Claude 要调某工具（`.name` / `.input` / `.id`） |
| `ToolResultBlock` | 工具返回结果（`.tool_use_id` / `.content` / `.is_error`） |

另有两个 server 端块偶尔出现：`ServerToolUseBlock` / `ServerToolResultBlock`（服务端执行的工具，如 advisor tool 的调用与结果）。做通用渲染时给一个兜底分支即可。

**一条消息能同时装好几件事**：一条 `AssistantMessage` 可能同时含"思考 + 文字 + 要调工具"。按块类型 `isinstance` 分流，就能重建出 agent 的完整行为轨迹（想什么→说什么→调什么→拿到什么）。按需求选处理层级：只要结果读 `ResultMessage`；要进度读 `AssistantMessage`（后台任务的进度另看 §7.2「SystemMessage 家族」 的 Task 系消息）；要打字机效果开 `include_partial_messages` 读 `StreamEvent`（§17「流式输出」）。下方代码 cell 就是这套分流的最小实现，运行一次即可在本页拿到属于自己的真实消息流。

In [1]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    SystemMessage,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ThinkingBlock,
    ToolUseBlock,
    ToolResultBlock,
)


async def demo_read_stream():
    session_id = None
    async for msg in query(
        prompt="List the files here, then summarize the project in one sentence.",
        options=ClaudeAgentOptions(
            cwd=".", allowed_tools=["Bash", "Glob", "Read"], max_turns=4
        ),
    ):
        if isinstance(msg, SystemMessage) and msg.subtype == "init":
            session_id = msg.data["session_id"]  # 记下会话 ID，§13 resume 要用
            print("session:", session_id)
        elif isinstance(msg, AssistantMessage):
            for b in msg.content:
                if isinstance(b, ThinkingBlock):
                    print("[thinking]", b.thinking[:80])
                elif isinstance(b, TextBlock):
                    print("[text]", b.text)
                elif isinstance(b, ToolUseBlock):
                    print("[tool_use]", b.name, b.input)
                elif isinstance(b, ToolResultBlock):
                    print("[tool_result]", str(b.content)[:80])
        elif isinstance(msg, ResultMessage):
            print(
                "[done]",
                "ok" if not msg.is_error else "ERROR",
                "turns=",
                msg.num_turns,
                "cost$=",
                msg.total_cost_usd,
            )
            print("final:", msg.result if msg.subtype == "success" else msg.subtype)


await demo_read_stream()

session: da19f712-a499-4073-b8e4-6dfe06706d0c
[thinking] 
[tool_use] Bash {'command': 'ls -la', 'description': 'List files in current directory'}
[text] **文件列表**（29 项，主体为 24 个 Jupyter notebook + 2 个 Markdown）：

| 文件 | 说明 |
|---|---|
| `00-总览与全局记忆图.ipynb` | 总纲 |
| `01-02-架构定位与底层原理.ipynb` | 架构定位 |
| `03-安装与认证.ipynb` | 安装认证 |
| `04-两个入口与两种输入模式.ipynb` | query / client 入口 |
| `05-最小可运行示例-query.ipynb` | 最小示例 |
| `06-Agent-loop-工作原理.ipynb` | Agent loop |
| `07-核心技能-读懂消息流.ipynb` | 消息流（最大，82 KB） |
| `08-配置总线-ClaudeAgentOptions.ipynb` | 配置 |
| `09-权限系统与Human-in-the-Loop.ipynb` | 权限 |
| `10-自定义工具-进程内MCP.ipynb` | 进程内 MCP |
| `11-Hooks-把agent变成可控系统.ipynb` | Hooks |
| `12-外部MCP-接入现成生态.ipynb` | 外部 MCP |
| `13-会话-continue-resume-fork.ipynb` | 会话管理 |
| `14-子Agent-把专项任务外包.ipynb` | 子 agent |
| `15-多轮与打断-ClaudeSDKClient.ipynb` | 多轮交互 |
| `16-流式输入…` / `17-流式输出…` | 流式 I/O |
| `18-结构化输出-验证过的JSON.ipynb` | 结构化输出 |
| `19-系统提示定制-四条路线.ipynb` | 系统提示 |
| `20-文件系统特性-setting_sources总开关.ipynb` | 文件系统配置 |
| `21-生产化-可运维